In [1]:
import ast

In [2]:
import csv

In [3]:
import pandas as pd

In [4]:
import numpy as np

In [5]:
import json

In [6]:
cyr = ['а','б','в','г','д','е','ё','ж','з','и','й','к','л','м','н','о','п','р','с','т','у','ф','х','ц','ч','ш','щ','ъ','ы','ь','э','ю','я','ґ','є','ї','ђ','љ','њ','ћ','џ', 'ў','ъ']
lat = ['a','b','v','g','d','e','jo','zh','z','i','j','k','l','m','n','o','p','r','s','t','u','f','h','ts','ch','sh','sczs','','y','','e','ju','ja','g','e','j','dzh','l','n','ch','dzh', 'w','o']
lat2 = ['ch','cz','rz','sz', 'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','qu','r','s','t','u','v','w','x', 'y','z','ą', 'ć','ę', 'ł','ń','ó','ś','ź','ż','á','č','ď','é','ě','í','ň','ř', 'š','ť','ú','ů','ý','ž','ä','ľ','ĺ', 'ô', 'ŕ','è']
cyr2 = ['х', 'ч', 'ж', 'ш',  'а','б','ц','д','е','ф','г','х','и','й','к','л','м','н','о','п','кв','р','с','т','у','в','в','кс','ы','з','ом','ч','ен','л','н','у','ш','ж','ж','а','ч','д','э','е','и','н','рж','ш','т','у','у','и','ж','э','л','лл','уо','рр','э']
langss = ['-', 'c', 'c', 'c', 'l', 'l', 'l', 'c', 'l']

# Transliterates text into proper script: 'c' - Cyrillic into Latinic, 'l' - conversely.
def transliterate(sss, l_kind):
  try:
    sss2 = sss.lower()

    if l_kind == 'c':
        l_from, l_to = cyr, lat
    elif l_kind == 'l':
        l_from, l_to = lat2, cyr2
    else:
        return ""
    for c, l in zip(l_from, l_to):
        try:
            sss2 = sss2.replace(c, l)
        except:
            pass
  except:
    pass
  return sss2

In [7]:
# 0.6 changes
phon_corr = {'о':['у'],
             'у':['о'],
             'е':['я', 'и', 'i', 'і'],
             'я':['е'],
             'д':['т'],
             'з':['с', 'ш'],
             'ж':['ш'],
             'к':['г'],
             'т':['д'],
             'с':['з', 'ш'],
             'ш':['ж'],
             'г':['к', 'х'],
             'и':['е'],
             'i': ['е'],
             'і': ['е'],
             'ч': ['ц'],
             'ц': ['ч'],
             'б': ['п'],
             'п': ['б'],
             'х': ['г'],
            }
# 0.5 changes
phon_corr_1 = {'ў':['в'],
               'в':['ў'],
               'й':['ј'],
               'ј':['й'],
               'и':['ј'],
               'ј':['и'],
               'њ':['н'],
               'н':['њ'],
               'љ':['л'],
               'л':['љ'],
               'ш':['щ'],
               'щ':['ш']
              }

# 0.2 changes
phon_corr_2 = {'ё':['е'],
               'е':['ё'],
               'i':['и', 'й'],
               'і': ['и', 'й'],
               'и':['i', 'ї', 'і', 'й'],
               'ї':['и', 'й'],
               'й':['i', 'і', 'ї', 'и'],
              }

# 0.3 changes
phon_corr_3 = {'о':['а'],
               'а':['о', 'я'],
               'я': ['а'],
               'ю': ['у'],
               'у': ['ю'],
               'е':['э'],         
               'ы':['и', 'i', 'і'],
               'и':['ы'],
               'i': ['ы'],
               'і': ['ы']
              }
# 1.2 changes: ћ, ђ and ъ with anything

def next_step_Levenshtein(dists, str1, str2, i, j):
    dists[i, j] = 1000
    v = min(dists[i-1, j-1], dists[i-1, j], dists[i, j-1])

    if str1[i-1] == 'ь' or str2[j-1] == 'ь':
        # "ь" в любой из строк считается за совпадение с чем угодно
        dists[i, j] = v  # Нулевая стоимость
        return
            
    if str1[i-1] == str2[j-1] and dists[i-1, j-1] <= dists[i-1, j] and dists[i-1, j-1] <= dists[i, j-1]:
        # print('+', str1[i-1: i+2], '|', str2[j-1: j+2], v)
        dists[i, j] = v
    else:
        # print('-', str1[i-1: i+2], '|', str2[j-1: j+2], v)
        if i < len(str1) - 1 and j < len(str2) - 1 and (str1[i-1], str1[i]) == (str2[j], str2[j-1]) and dists[i, j] > v + 0.5:
            # print("+")
            dists[i, j] = v + 0.5
        if i < len(str1) - 2 and j < len(str2) - 2 and (str1[i-2], str1[i-1]) == (str2[j-1], str2[j-2]) and dists[i, j] > v + 0.5:
            # print("++")
            dists[i, j] = v + 0.5
        if str1[i-1] in phon_corr.keys() and str2[j-1] in phon_corr[str1[i-1]] and dists[i, j] > v + 0.6:
            dists[i, j] = v + 0.6
        if str1[i-1] in phon_corr_1.keys() and str2[j-1] in phon_corr_1[str1[i-1]] and dists[i, j] > v + 0.5:
            dists[i, j] = v + 0.5
        if str1[i-1] in phon_corr_2.keys() and str2[j-1] in phon_corr_2[str1[i-1]] and dists[i, j] > v + 0.2:
            dists[i, j] = v + 0.2        
        if dists[i, j] > v + 1:
            dists[i, j] = v + 1

def find_path_Levenstein(dists, str1, str2):
    path = [] # [('>', '>', float(dists[len(str1)+1, len(str2)+1]))]
    pos1, pos2 = len(str1), len(str2)
    while pos1 > 0 and pos2 > 0:
        if dists[pos1, pos2] < dists[pos1, pos2+1] and dists[pos1, pos2] < dists[pos1+1, pos2]:
            path.append((str1[pos1-1], str2[pos2-1], float(dists[pos1, pos2])))
            pos1 -= 1
            pos2 -= 1
        elif dists[pos1+1, pos2] <= dists[pos1, pos2] and dists[pos1+1, pos2] <= dists[pos1+1, pos2]:
            path.append(('_', str2[pos2-1], float(dists[pos1, pos2])))
            pos2 -= 1
        else:
            path.append((str1[pos1-1], '_', float(dists[pos1, pos2])))
            pos1 -= 1

    if pos1 != 0:
        while pos1 != 0:
            path.append((str1[pos1-1], '_', float(dists[pos1, 0])))
            pos1 -= 1
    elif pos2 != 0:
        while pos2 != 0:
            path.append(('_', str2[pos2-1], float(dists[0, pos2])))
            pos2 -= 1
    return path

def fast_Levenstein(str1, str2):
    dists = np.zeros((len(str1)+2, len(str2)+2))
    dists[0, :] = np.arange(0, len(str2)+2, 1)
    dists[:, 0] = np.arange(0, len(str1)+2, 1)
    # print(dists)
    for i in range(1, len(str1)+1):
        for j in range(1, len(str2)+1):
            # print(i, j)
            # print(str1[i-1], str2[j-1])
            next_step_Levenshtein(dists, str1, str2, i, j)
            
        dists[i, len(str2)+1] = min(dists[i-1, len(str2)], dists[i-1, len(str2)+1], dists[i, len(str2)]) + 1
        # print(dists)
        
    for j in range(1, len(str2)+1):
        dists[len(str1)+1, j] = min(dists[len(str1), j-1], dists[len(str1), j], dists[len(str1)+1, j-1]) + 1
    dists[len(str1)+1, len(str2)+1] = dists[len(str1), len(str2)]
    # print(dists)
    path = find_path_Levenstein(dists, str1, str2)
    # print(path[::-1])
    return path[::-1][-1][-1]

In [8]:
russian_true = pd.read_csv('complete_morphs/russian_morphs.csv')
answers_true = pd.read_csv('complete_morphs/user_answers_morphs.csv')
ukrainian_true = pd.read_csv('complete_morphs/ukrainian_morphs.csv')
belarus_true = pd.read_csv('complete_morphs/belarus_morphs.csv')
bulgarian_true = pd.read_csv('complete_morphs/bulgarian_morphs.csv')
polish_true = pd.read_csv('complete_morphs/polish_morphs.csv')
czech_true = pd.read_csv('complete_morphs/czech_morphs.csv')
serbian_true = pd.read_csv('complete_morphs/serbian_morphs.csv')
slovak_true = pd.read_csv('complete_morphs/slovak_morphs.csv')
slovene_true = pd.read_csv('complete_morphs/slovene_morphs.csv')

In [9]:
ukrainian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Ukranian']
belarus_ans = answers_true.loc[answers_true['parallel_lang'] == 'Belarussian']
czech_ans = answers_true.loc[answers_true['parallel_lang'] == 'Czech']
polish_ans = answers_true.loc[answers_true['parallel_lang'] == 'Polish']
bulgarian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Bulgarian']
serbian_ans = answers_true.loc[answers_true['parallel_lang'] == 'Serbian']
slovak_ans = answers_true.loc[answers_true['parallel_lang'] == 'Slovak']
slovene_ans = answers_true.loc[answers_true['parallel_lang'] == 'Slovene']
nopar_ans = answers_true.loc[answers_true['parallel_lang'] == 'No Parallel Text']

In [10]:
num_type = {}

for _, row in nopar_ans.iterrows():
    test_id = row['test_id']
    answer_no = row['answer_no']
    correctness = row['correctness']
    code = (int(f'{test_id}{answer_no}'))
    if code not in num_type.keys():
        num_type[code] = {}
    if correctness not in num_type[code].keys():
        num_type[code][correctness] = 1
    else:
        num_type[code][correctness] +=1
for key in num_type:    
    sum_ans = 0
    for num in num_type[key]:
        sum_ans += num_type[key][num]
    if 'incorr' in num_type[key]:
        num_type[key]['mistakes'] = num_type[key]['incorr'] / sum_ans
    else:
        num_type[key]['mistakes'] = 0

for key in num_type:
    num_type[key] = dict(sorted(num_type[key].items(), key=lambda item: item[1], reverse=True))

num_type

{10: {'corr': 19, 'pcorr': 18, 'incorr': 7, 'mistakes': 0.1590909090909091},
 11: {'pcorr': 30, 'corr': 10, 'incorr': 4, 'mistakes': 0.09090909090909091},
 12: {'pcorr': 29, 'corr': 12, 'incorr': 3, 'mistakes': 0.06818181818181818},
 13: {'corr': 35, 'pcorr': 6, 'incorr': 3, 'mistakes': 0.06818181818181818},
 14: {'incorr': 23, 'pcorr': 17, 'corr': 4, 'mistakes': 0.5227272727272727},
 15: {'corr': 24, 'pcorr': 10, 'incorr': 10, 'mistakes': 0.22727272727272727},
 16: {'corr': 34, 'pcorr': 5, 'incorr': 5, 'mistakes': 0.11363636363636363},
 17: {'pcorr': 27, 'corr': 16, 'incorr': 1, 'mistakes': 0.022727272727272728},
 18: {'pcorr': 24, 'incorr': 13, 'corr': 7, 'mistakes': 0.29545454545454547},
 19: {'pcorr': 30, 'corr': 10, 'incorr': 4, 'mistakes': 0.09090909090909091},
 110: {'corr': 26, 'pcorr': 16, 'incorr': 2, 'mistakes': 0.045454545454545456},
 111: {'pcorr': 34, 'incorr': 5, 'corr': 5, 'mistakes': 0.11363636363636363},
 112: {'corr': 34, 'pcorr': 7, 'incorr': 3, 'mistakes': 0.068181

In [11]:
def cognate_searcher(answer_set, russian_true, language):
    result_dict = {}
    for _, row in answer_set.iterrows():
        test_id = row['test_id']
        answer_no = row['answer_no']
        user_answer = str(row['answer']).lower().strip()
        correctness = row['correctness']
        code_ans = (user_answer, correctness)
        
        correct_row = russian_true[
            (russian_true['test_id'] == test_id) & 
            (russian_true['answer_no'] == answer_no)
        ]
        original_row = language[
            (language['test_id'] == test_id) & 
            (language['answer_no'] == answer_no)
        ]
        
        if not correct_row.empty:
            correct_answer = str(correct_row.iloc[0]['answer']).strip()
            original_answer = str(original_row.iloc[0]['answer']).strip()
            code = (int(f'{test_id}{answer_no}'), correct_answer, original_answer)
            if code not in result_dict.keys():
                result_dict[code] = {}
        if code_ans not in result_dict[code].keys():
            result_dict[code][code_ans] = 1
        else:
            result_dict[code][code_ans] +=1
    
    for key in result_dict:
        result_dict[key] = dict(sorted(result_dict[key].items(), key=lambda item: item[1], reverse=True))

    return result_dict

In [12]:
ukr_cognates = cognate_searcher(ukrainian_ans, russian_true, ukrainian_true)
ukr_cognates

{(10, 'повернулся', 'повернувся'): {('повернулся', 'corr'): 35,
  ('развернулся', 'corr'): 1,
  ('вернулся', 'pcorr'): 1,
  ('пошёл', 'pcorr'): 1},
 (11, 'ступеням', 'сходинок'): {('ступенькам', 'corr'): 6,
  ('nan', 'incorr'): 5,
  ('ступеням', 'corr'): 4,
  ('сходням', 'corr'): 4,
  ('лестнице', 'corr'): 3,
  ('сходинок', 'incorr'): 2,
  ('выходу', 'pcorr'): 2,
  ('сходне', 'pcorr'): 1,
  ('сходу', 'incorr'): 1,
  ('перекрестку', 'pcorr'): 1,
  ('лестницам', 'corr'): 1,
  ('воротам', 'pcorr'): 1,
  ('сходке', 'pcorr'): 1,
  ('колоннам', 'pcorr'): 1,
  ('приступку', 'pcorr'): 1,
  ('входу', 'pcorr'): 1,
  ('середине', 'pcorr'): 1,
  ('дорожкам', 'pcorr'): 1,
  ('пркступникам', 'incorr'): 1},
 (12, 'шашек', 'шашок'): {('шашек', 'corr'): 14,
  ('nan', 'incorr'): 7,
  ('досок', 'pcorr'): 5,
  ('плиток', 'corr'): 5,
  ('шашок', 'corr'): 2,
  ('цветов', 'pcorr'): 1,
  ('плит', 'corr'): 1,
  ('каамней', 'pcorr'): 1,
  ('шапок', 'incorr'): 1,
  ('узоров', 'corr'): 1},
 (13, 'спиною', 'спиною

In [13]:
bel_cognates = cognate_searcher(belarus_ans, russian_true, belarus_true)
bel_cognates

{(10, 'повернулся', 'завярнуўся'): {('развернулся', 'corr'): 17,
  ('повернулся', 'corr'): 10,
  ('обернулся', 'pcorr'): 3,
  ('завернулся', 'incorr'): 2,
  ('оглянулся', 'pcorr'): 2,
  ('завернул', 'pcorr'): 1,
  ('равзвернулся', 'corr'): 1,
  ('завернулся в плащ', 'pcorr'): 1,
  ('отвернулся', 'corr'): 1},
 (11, 'ступеням', 'прыступак'): {('ступеням', 'corr'): 9,
  ('помосту', 'pcorr'): 5,
  ('nan', 'incorr'): 4,
  ('крыльцу', 'pcorr'): 3,
  ('преступнику', 'pcorr'): 3,
  ('приступку', 'pcorr'): 2,
  ('лестнице', 'corr'): 2,
  ('пристани', 'pcorr'): 2,
  ('тупику', 'pcorr'): 1,
  ('причалу', 'pcorr'): 1,
  ('дверям', 'pcorr'): 1,
  ('помосту и ступеням', 'corr'): 1,
  ('воротам', 'pcorr'): 1,
  ('ступенькам', 'corr'): 1,
  ('приступкам', 'incorr'): 1,
  ('обрыву', 'pcorr'): 1},
 (12, 'шашек', 'шашак'): {('шашек', 'corr'): 7,
  ('плиток', 'corr'): 5,
  ('nan', 'incorr'): 3,
  ('досок', 'pcorr'): 3,
  ('деталей', 'pcorr'): 2,
  ('мозаик', 'corr'): 2,
  ('кусочков', 'corr'): 1,
  ('кусо

In [14]:
bg_cognates = cognate_searcher(bulgarian_ans, russian_true, bulgarian_true)
bg_cognates

{(10, 'повернулся', 'обърна'): {('обернулся', 'pcorr'): 12,
  ('повернулся', 'corr'): 12,
  ('развернулся', 'corr'): 3,
  ('оглянулся', 'pcorr'): 2,
  ('встал', 'pcorr'): 2,
  ('nan', 'incorr'): 2,
  ('прошелся туда и обратно по подиуму', 'incorr'): 1,
  ('василий', 'incorr'): 1,
  ('прыгнул с обрыва', 'incorr'): 1,
  ('огляделся', 'pcorr'): 1},
 (11, 'ступеням', 'стъпалата'): {('nan', 'incorr'): 10,
  ('ступеням', 'corr'): 7,
  ('палате', 'incorr'): 4,
  ('дому', 'pcorr'): 3,
  ('площади', 'pcorr'): 2,
  ('като', 'incorr'): 1,
  ('собору', 'pcorr'): 1,
  ('трибуне', 'pcorr'): 1,
  ('толпе', 'pcorr'): 1,
  ('ступени', 'corr'): 1,
  ('ступенькам', 'corr'): 1,
  ('храму', 'pcorr'): 1,
  ('лестнице', 'corr'): 1,
  ('креслу', 'pcorr'): 1,
  ('палатке', 'pcorr'): 1,
  ('бордюру', 'pcorr'): 1},
 (12, 'шашек', 'плочки'): {('плиток', 'corr'): 9,
  ('досок', 'pcorr'): 8,
  ('шашек', 'corr'): 3,
  ('камней', 'pcorr'): 3,
  ('дощечек', 'pcorr'): 3,
  ('шахматных плиток', 'corr'): 2,
  ('nan', 'in

In [15]:
cz_cognates = cognate_searcher(czech_ans, russian_true, czech_true)
cz_cognates

{(10, 'повернулся', 'otočil'): {('обернулся', 'pcorr'): 8,
  ('развернулся', 'corr'): 5,
  ('отвернулся', 'corr'): 4,
  ('nan', 'incorr'): 3,
  ('замолчал', 'pcorr'): 2,
  ('встал', 'pcorr'): 2,
  ('закончил', 'pcorr'): 2,
  ('посмотрел', 'pcorr'): 1,
  ('повернулся', 'corr'): 1,
  ('покричал', 'pcorr'): 1,
  ('отошёл', 'pcorr'): 1,
  ('повернул', 'corr'): 1,
  ('пригнулся', 'pcorr'): 1,
  ('оглянулся', 'pcorr'): 1,
  ('отскочил', 'pcorr'): 1,
  ('отскочил от края', 'pcorr'): 1,
  ('оточился', 'incorr'): 1,
  ('отошел', 'pcorr'): 1},
 (11, 'ступеням', 'stupňům'): {('ступеням', 'corr'): 24,
  ('ступенькам', 'corr'): 2,
  ('дороге', 'pcorr'): 2,
  ('замку', 'pcorr'): 2,
  ('nan', 'incorr'): 2,
  ('ступенька', 'corr'): 1,
  ('лестнице', 'corr'): 1,
  ('доиу', 'pcorr'): 1,
  ('трибуне', 'pcorr'): 1,
  ('месту', 'pcorr'): 1},
 (12, 'шашек', 'čtverce'): {('nan', 'incorr'): 5,
  ('камней', 'pcorr'): 4,
  ('квадратов', 'corr'): 4,
  ('досок', 'pcorr'): 3,
  ('плиток', 'corr'): 3,
  ('цветов', 

In [16]:
pol_cognates = cognate_searcher(polish_ans, russian_true, polish_true)
pol_cognates

{(10, 'повернулся', 'odwrócił'): {('повернулся', 'corr'): 11,
  ('развернулся', 'corr'): 4,
  ('оглянулся', 'pcorr'): 3,
  ('встал', 'pcorr'): 2,
  ('обернулся', 'pcorr'): 2,
  ('отвернулся', 'corr'): 2,
  ('подумал', 'pcorr'): 1,
  ('откинул это', 'incorr'): 1,
  ('взял', 'corr'): 1,
  ('прикрыл глаза', 'pcorr'): 1,
  ('открыл дверь', 'pcorr'): 1,
  ('nan', 'incorr'): 1,
  ('поверил в это', 'incorr'): 1,
  ('одбросил', 'incorr'): 1,
  ('повернулся назад', 'corr'): 1,
  ('одумался', 'incorr'): 1,
  ('оглядел его', 'pcorr'): 1,
  ('вздохнул', 'pcorr'): 1},
 (11, 'ступеням', 'schodów'): {('лестнице', 'corr'): 9,
  ('ступеням', 'corr'): 4,
  ('nan', 'incorr'): 3,
  ('выходу', 'pcorr'): 3,
  ('замку', 'pcorr'): 2,
  ('спуску', 'pcorr'): 1,
  ('дороге', 'pcorr'): 1,
  ('сторону сходней', 'incorr'): 1,
  ('лестнице, спуску', 'corr'): 1,
  ('реке', 'pcorr'): 1,
  ('школе', 'pcorr'): 1,
  ('судам', 'pcorr'): 1,
  ('плахе', 'pcorr'): 1,
  ('обрыву', 'pcorr'): 1,
  ('оврагу', 'pcorr'): 1,
  ('то

In [17]:
sn_cognates = cognate_searcher(slovene_ans, russian_true, slovene_true)
sn_cognates

{(10, 'повернулся', 'zasukal'): {('повернулся', 'corr'): 10,
  ('nan', 'incorr'): 7,
  ('заскучал', 'pcorr'): 4,
  ('развернулся', 'corr'): 3,
  ('повернул', 'corr'): 1,
  ('подумал', 'pcorr'): 1,
  ('понял', 'incorr'): 1,
  ('замолчал', 'pcorr'): 1,
  ('увидел', 'pcorr'): 1,
  ('встал', 'pcorr'): 1,
  ('вышел', 'pcorr'): 1,
  ('посмотрел', 'pcorr'): 1,
  ('задумался', 'pcorr'): 1,
  ('павернулся', 'corr'): 1,
  ('сказал', 'incorr'): 1,
  ('постоял', 'pcorr'): 1,
  ('застукал', 'incorr'): 1},
 (11, 'ступеням', 'stopnicam'): {('лестнице', 'corr'): 12,
  ('ступеням', 'corr'): 8,
  ('nan', 'incorr'): 4,
  ('ступенькам', 'corr'): 2,
  ('воротам', 'pcorr'): 2,
  ('дворцу', 'pcorr'): 2,
  ('горе', 'pcorr'): 1,
  ('сходням', 'corr'): 1,
  ('себе', 'incorr'): 1,
  ('гостям', 'pcorr'): 1,
  ('стенам', 'pcorr'): 1,
  ('нему', 'incorr'): 1,
  ('стопницам', 'incorr'): 1},
 (12, 'шашек', 'ploščic'): {('квадратов', 'corr'): 10,
  ('плиток', 'corr'): 5,
  ('узоров', 'corr'): 4,
  ('nan', 'incorr'): 3

In [18]:
sk_cognates = cognate_searcher(slovak_ans, russian_true, slovak_true)
sk_cognates

{(10, 'повернулся', 'sa obrátil'): {('повернулся', 'corr'): 20,
  ('обернулся', 'pcorr'): 16,
  ('развернулся', 'corr'): 11,
  ('nan', 'incorr'): 3,
  ('обратился', 'incorr'): 2,
  ('повернул', 'corr'): 2,
  ('сообразил', 'incorr'): 2,
  ('осмотрелся', 'incorr'): 2,
  ('оглянулся', 'pcorr'): 1,
  ('обратился к скрещенной трибуне', 'incorr'): 1,
  ('обратил', 'incorr'): 1,
  ('вышел', 'pcorr'): 1,
  ('спустился', 'pcorr'): 1,
  ('повернулся(развернулся)', 'corr'): 1,
  ('заметил', 'incorr'): 1,
  ('посмотрел', 'pcorr'): 1,
  ('повернуля', 'corr'): 1,
  ('взял', 'corr'): 1,
  ('собрался', 'pcorr'): 1},
 (11, 'ступеням', 'schodíkom'): {('ступеням', 'corr'): 15,
  ('nan', 'incorr'): 9,
  ('ступенькам', 'corr'): 5,
  ('лестнице', 'corr'): 5,
  ('спуску', 'pcorr'): 4,
  ('трибуне', 'pcorr'): 4,
  ('воротам', 'pcorr'): 3,
  ('помосту', 'pcorr'): 3,
  ('выходу', 'pcorr'): 2,
  ('писарю', 'pcorr'): 1,
  ('не глядя ни направо ни налево', 'incorr'): 1,
  ('дворцу', 'pcorr'): 1,
  ('сходням', 'cor

In [19]:
sb_cognates = cognate_searcher(serbian_ans, russian_true, serbian_true)
sb_cognates

{(10, 'повернулся', 'окренуо'): {('повернулся', 'corr'): 9,
  ('оглянулся', 'pcorr'): 9,
  ('обернулся', 'pcorr'): 5,
  ('nan', 'incorr'): 4,
  ('развернулся', 'corr'): 2,
  ('поклонился', 'pcorr'): 1,
  ('повернул', 'corr'): 1,
  ('нагнулся', 'pcorr'): 1,
  ('наклонился', 'pcorr'): 1,
  ('поднялся', 'pcorr'): 1,
  ('окрикнул', 'incorr'): 1,
  ('встал', 'pcorr'): 1,
  ('подумал', 'pcorr'): 1,
  ('повернулся вокруг', 'corr'): 1,
  ('огляделся', 'pcorr'): 1,
  ('онлянулся', 'pcorr'): 1},
 (11, 'ступеням', 'степеницама'): {('лестнице', 'corr'): 13,
  ('ступеням', 'corr'): 6,
  ('платформе', 'pcorr'): 5,
  ('nan', 'incorr'): 5,
  ('помосту', 'pcorr'): 3,
  ('ступенькам', 'corr'): 2,
  ('воротам', 'pcorr'): 2,
  ('плаформе', 'pcorr'): 1,
  ('площади', 'pcorr'): 1,
  ('темницам', 'pcorr'): 1,
  ('деревне', 'pcorr'): 1},
 (12, 'шашек', 'коцака'): {('nan', 'incorr'): 11,
  ('кубиков', 'pcorr'): 5,
  ('досок', 'pcorr'): 4,
  ('камней', 'pcorr'): 4,
  ('плит', 'corr'): 3,
  ('плиток', 'corr'): 3

In [20]:
def morph_comparer(set_ru, set_cg, orpho):
    
    def compare_morphemes(ru_list, cg_list, morpheme_type):

        scores = []
        pairs = []

        if not ru_list and not cg_list:
            return {'scores': scores, 'pairs': pairs}

        if not ru_list:
            cg_combined = ''.join(cg_list)
            scores.append(fast_Levenstein('', cg_combined))
            pairs.append(('', cg_combined))
            return {'scores': scores, 'pairs': pairs}

        if not cg_list:
            ru_combined = ''.join(ru_list)
            scores.append(fast_Levenstein(ru_combined, ''))
            pairs.append((ru_combined, ''))
            return {'scores': scores, 'pairs': pairs}

        if len(ru_list) == len(cg_list):
            for ru_elem, cg_elem in zip(ru_list, cg_list):
                scores.append(fast_Levenstein(ru_elem, cg_elem))
                pairs.append((ru_elem, cg_elem))
            return {'scores': scores, 'pairs': pairs}

        if morpheme_type in ['SUFF', 'PREF']:
            return {'scores': ['HANDCHECK'], 'pairs': [ru_list, cg_list]}

    result = {}

    ru_dict = ast.literal_eval(set_ru)
    cg_dict = ast.literal_eval(set_cg)

    if orpho == 'latin':
        if isinstance(cg_dict, dict):
            for key in cg_dict:
                cg_dict[key] = [transliterate(el, 'l') for el in cg_dict[key]]

    morpheme_types = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
    not_link = 0

    for morpheme_type in morpheme_types:

        ru_list = []
        cg_list = []

        if isinstance(ru_dict, dict) and isinstance(cg_dict, dict):
            ru_list = ru_dict.get(morpheme_type, [])
            cg_list = cg_dict.get(morpheme_type, [])

        if not ru_list and not cg_list:
            continue

        if morpheme_type == 'ROOT' and len(ru_list) > len(cg_list) and len(cg_list) != 0:
            not_link = 1
            if 'LINK' in ru_dict.keys():
                ru_root = ru_list[0] + ru_dict['LINK'][0] + ru_list[1]
            else:
                ru_root = ru_list[0] + ru_list[1]
            result[morpheme_type] = {'scores': [fast_Levenstein(ru_root, cg_list[0])], 'pairs': [(ru_root, cg_list[0])]}
            continue

        if morpheme_type == 'ROOT' and len(cg_list) > len(ru_list) and len(ru_list) != 0:
            not_link = 1
            if 'LINK' in cg_dict.keys():
                cg_root = cg_list[0] + cg_dict['LINK'][0] + cg_list[1]
            else:
                cg_root = cg_list[0] + cg_list[1]
            result[morpheme_type] = {'scores': [fast_Levenstein(ru_list[0], cg_root)], 'pairs': [(ru_list[0], cg_root)]}
            continue

        if not not_link:
            result[morpheme_type] = compare_morphemes(ru_list, cg_list, morpheme_type)

    return result

In [21]:
def cognate_stats(answer_set, russian_true, language, language_dict, orpho, num_type):
    true_cognates = {}
    part_cognates = {}
    for _, row in answer_set.iterrows():
        test_id = row['test_id']
        answer_no = row['answer_no']
        user_answer = str(row['answer']).lower().strip()
        correctness = row['correctness']
        code_ans = (user_answer, correctness)
        morphs_ans = row['morphemic_parse']

        correct_row = russian_true[
            (russian_true['test_id'] == test_id) &
            (russian_true['answer_no'] == answer_no)
            ]
        original_row = language[
            (language['test_id'] == test_id) &
            (language['answer_no'] == answer_no)
            ]

        if not correct_row.empty:
            correct_answer = str(correct_row.iloc[0]['answer']).strip()
            original_answer = str(original_row.iloc[0]['answer']).strip()
            correct_morph = str(correct_row.iloc[0]['morphemic_parse']).strip()
            original_morph = str(original_row.iloc[0]['morphemic_parse']).strip()
            code = (int(f'{test_id}{answer_no}'), correct_answer, original_answer)
            if float(original_row.iloc[0]['cognate']) != 0:
                if float(original_row.iloc[0]['cognate']) == 1:
                    if code not in true_cognates.keys():
                        true_cognates[code] = {}
                        true_cognates[code]['morph_comparison'] = morph_comparer(correct_morph, original_morph, orpho)
                        true_cognates[code]['overall_stats'] = {}
                        true_cognates[code]['word_stats'] = {}

                elif code not in part_cognates.keys():
                    part_cognates[code] = {}
                    part_cognates[code]['morph_comparison'] = morph_comparer(correct_morph, original_morph, orpho)
                    part_cognates[code]['overall_stats'] = {}
                    part_cognates[code]['word_stats'] = {}

        num_ans = 0
        for key in language_dict[code]:
            num_ans += language_dict[code][key]

        if float(original_row.iloc[0]['cognate']) == 1:
            if correctness not in true_cognates[code]['overall_stats'].keys():
                counter = 0
                for key in language_dict[code]:
                    if key[1] == correctness:
                        counter += language_dict[code][key]
                true_cognates[code]['overall_stats'][correctness] = counter / num_ans
                if correctness == 'incorr':
                    mistakes_control = num_type[code[0]]['mistakes']
                    if mistakes_control != 0:
                        explained_mistakes = (mistakes_control - true_cognates[code]['overall_stats'][
                            correctness]) / mistakes_control
                    else:
                        explained_mistakes = 0 - true_cognates[code]['overall_stats'][correctness]
                    true_cognates[code]['overall_stats']['explained_mistakes'] = explained_mistakes

            if code_ans not in true_cognates[code]['word_stats'].keys():
                true_cognates[code]['word_stats'][code_ans] = (
                morph_comparer(morphs_ans, original_morph, orpho), language_dict[code][code_ans] / num_ans)

        elif float(original_row.iloc[0]['cognate']) == 0.5:
            if correctness not in part_cognates[code]['overall_stats'].keys():
                counter = 0
                for key in language_dict[code]:
                    if key[1] == correctness:
                        counter += language_dict[code][key]
                part_cognates[code]['overall_stats'][correctness] = counter / num_ans
                if correctness == 'incorr':
                    mistakes_control = num_type[code[0]]['mistakes']
                    if mistakes_control != 0:
                        explained_mistakes = (mistakes_control - part_cognates[code]['overall_stats'][
                            correctness]) / mistakes_control
                    else:
                        explained_mistakes = 0 - part_cognates[code]['overall_stats'][correctness]
                    part_cognates[code]['overall_stats']['explained_mistakes'] = explained_mistakes
            if code_ans not in part_cognates[code]['word_stats'].keys():
                part_cognates[code]['word_stats'][code_ans] = (
                morph_comparer(morphs_ans, original_morph, orpho), language_dict[code][code_ans] / num_ans)

    for key in true_cognates:
        for el in true_cognates[key]:
            if el == 'overall_stats':
                true_cognates[key][el] = dict(
                    sorted(true_cognates[key][el].items(), key=lambda item: item[1], reverse=True))
            elif el == 'word_stats':
                true_cognates[key][el] = dict(
                    sorted(true_cognates[key][el].items(), key=lambda item: item[1][1], reverse=True))

    for key in part_cognates:
        for el in part_cognates[key]:
            if el == 'overall_stats':
                part_cognates[key][el] = dict(
                    sorted(part_cognates[key][el].items(), key=lambda item: item[1], reverse=True))
            elif el == 'word_stats':
                part_cognates[key][el] = dict(
                    sorted(part_cognates[key][el].items(), key=lambda item: item[1][1], reverse=True))

    return true_cognates, part_cognates

In [22]:
ukr_stats = cognate_stats(ukrainian_ans, russian_true, ukrainian_true, ukr_cognates, 'cyr', num_type)
ukr_stats

({(10,
   'повернулся',
   'повернувся'): {'morph_comparison': {'PREF': {'scores': [0.0],
     'pairs': [('по', 'по')]},
    'ROOT': {'scores': [0.0], 'pairs': [('вер', 'вер')]},
    'SUFF': {'scores': [0.0, 0.0, 1.0],
     'pairs': [('н', 'н'), ('у', 'у'), ('л', 'в')]},
    'POSTFIX': {'scores': [0.0],
     'pairs': [('ся', 'ся')]}}, 'overall_stats': {'corr': 0.9473684210526315,
    'pcorr': 0.05263157894736842}, 'word_stats': {('повернулся',
     'corr'): ({'PREF': {'scores': [0.0], 'pairs': [('по', 'по')]},
      'ROOT': {'scores': [0.0], 'pairs': [('вер', 'вер')]},
      'SUFF': {'scores': ['HANDCHECK'],
       'pairs': [['ну', 'л'], ['н', 'у', 'в']]},
      'POSTFIX': {'scores': [0.0], 'pairs': [('ся', 'ся')]}},
     0.9210526315789473),
    ('развернулся',
     'corr'): ({'PREF': {'scores': [3.0], 'pairs': [('раз', 'по')]},
      'ROOT': {'scores': [0.0], 'pairs': [('вер', 'вер')]},
      'SUFF': {'scores': [0.0, 0.0, 1.0],
       'pairs': [('н', 'н'), ('у', 'у'), ('л', 'в')]},
 

In [23]:
bel_stats = cognate_stats(belarus_ans, russian_true, belarus_true, bel_cognates, 'cyr', num_type)
bel_stats

({(10,
   'повернулся',
   'завярнуўся'): {'morph_comparison': {'PREF': {'scores': [2.0],
     'pairs': [('по', 'за')]},
    'ROOT': {'scores': [0.6], 'pairs': [('вер', 'вяр')]},
    'SUFF': {'scores': [0.0, 0.0, 1.0],
     'pairs': [('н', 'н'), ('у', 'у'), ('л', 'ў')]},
    'POSTFIX': {'scores': [0.0],
     'pairs': [('ся', 'ся')]}}, 'overall_stats': {'corr': 0.7631578947368421,
    'explained_mistakes': 0.6691729323308271,
    'pcorr': 0.18421052631578946,
    'incorr': 0.05263157894736842}, 'word_stats': {('развернулся',
     'corr'): ({'PREF': {'scores': [2.0], 'pairs': [('раз', 'за')]},
      'ROOT': {'scores': [0.6], 'pairs': [('вер', 'вяр')]},
      'SUFF': {'scores': [0.0, 0.0, 1.0],
       'pairs': [('н', 'н'), ('у', 'у'), ('л', 'ў')]},
      'POSTFIX': {'scores': [0.0], 'pairs': [('ся', 'ся')]}},
     0.4473684210526316),
    ('повернулся',
     'corr'): ({'PREF': {'scores': [2.0], 'pairs': [('по', 'за')]},
      'ROOT': {'scores': [0.6], 'pairs': [('вер', 'вяр')]},
      'SU

In [24]:
cz_stats = cognate_stats(czech_ans, russian_true, czech_true, cz_cognates, 'latin', num_type)
cz_stats

({(11,
   'ступеням',
   'stupňům'): {'morph_comparison': {'ROOT': {'scores': [1.0],
     'pairs': [('ступен', 'ступн')]},
    'END': {'scores': [1.0],
     'pairs': [('ям', 'ум')]}}, 'overall_stats': {'corr': 0.7567567567567568,
    'explained_mistakes': 0.4054054054054054,
    'pcorr': 0.1891891891891892,
    'incorr': 0.05405405405405406}, 'word_stats': {('ступеням',
     'corr'): ({'ROOT': {'scores': [1.0], 'pairs': [('ступен', 'ступн')]},
      'END': {'scores': [1.0], 'pairs': [('ям', 'ум')]}},
     0.6486486486486487),
    ('ступенькам',
     'corr'): ({'ROOT': {'scores': [1.0], 'pairs': [('ступ', 'ступн')]},
      'SUFF': {'scores': [4.0], 'pairs': [('еньк', '')]},
      'END': {'scores': [1.0], 'pairs': [('ам', 'ум')]}}, 0.05405405405405406),
    ('дороге',
     'pcorr'): ({'ROOT': {'scores': [5.0], 'pairs': [('дорог', 'ступн')]},
      'END': {'scores': [2.0], 'pairs': [('е', 'ум')]}}, 0.05405405405405406),
    ('замку',
     'pcorr'): ({'ROOT': {'scores': [4.6], 'pairs': [('

In [25]:
pol_stats = cognate_stats(polish_ans, russian_true, polish_true, pol_cognates, 'latin', num_type)
pol_stats

({(10,
   'повернулся',
   'odwrócił'): {'morph_comparison': {'PREF': {'scores': [2.0],
     'pairs': [('по', 'од')]},
    'ROOT': {'scores': [3.0], 'pairs': [('вер', 'вруц')]},
    'SUFF': {'scores': ['HANDCHECK'], 'pairs': [['н', 'у', 'л'], ['и', 'л']]},
    'POSTFIX': {'scores': [2.0],
     'pairs': [('ся', '')]}}, 'overall_stats': {'corr': 0.5277777777777778,
    'pcorr': 0.3333333333333333,
    'incorr': 0.1388888888888889,
    'explained_mistakes': 0.12698412698412692}, 'word_stats': {('повернулся',
     'corr'): ({'PREF': {'scores': [2.0], 'pairs': [('по', 'од')]},
      'ROOT': {'scores': [3.0], 'pairs': [('вер', 'вруц')]},
      'SUFF': {'scores': [2.0, 0.0], 'pairs': [('ну', 'и'), ('л', 'л')]},
      'POSTFIX': {'scores': [2.0], 'pairs': [('ся', '')]}},
     0.3055555555555556),
    ('развернулся',
     'corr'): ({'PREF': {'scores': [3.0], 'pairs': [('раз', 'од')]},
      'ROOT': {'scores': [3.0], 'pairs': [('вер', 'вруц')]},
      'SUFF': {'scores': ['HANDCHECK'],
       'pa

In [26]:
bg_stats = cognate_stats(bulgarian_ans, russian_true, bulgarian_true, bg_cognates, 'cyr', num_type)
bg_stats

({(10,
   'повернулся',
   'обърна'): {'morph_comparison': {'PREF': {'scores': [2.0],
     'pairs': [('по', 'об')]},
    'ROOT': {'scores': [2.0], 'pairs': [('вер', 'ър')]},
    'SUFF': {'scores': ['HANDCHECK'], 'pairs': [['н', 'у', 'л'], ['н']]},
    'END': {'scores': [1.0], 'pairs': [('', 'а')]},
    'POSTFIX': {'scores': [2.0],
     'pairs': [('ся', '')]}}, 'overall_stats': {'pcorr': 0.4594594594594595,
    'corr': 0.40540540540540543,
    'explained_mistakes': 0.15057915057915053,
    'incorr': 0.13513513513513514}, 'word_stats': {('обернулся',
     'pcorr'): ({'PREF': {'scores': [1.0], 'pairs': [('о', 'об')]},
      'ROOT': {'scores': [2.0], 'pairs': [('бер', 'ър')]},
      'SUFF': {'scores': ['HANDCHECK'], 'pairs': [['н', 'у', 'л'], ['н']]},
      'END': {'scores': [1.0], 'pairs': [('', 'а')]},
      'POSTFIX': {'scores': [2.0], 'pairs': [('ся', '')]}},
     0.32432432432432434),
    ('повернулся',
     'corr'): ({'PREF': {'scores': [2.0], 'pairs': [('по', 'об')]},
      'ROOT': 

In [27]:
sb_stats = cognate_stats(serbian_ans, russian_true, serbian_true, sb_cognates, 'cyr', num_type)
sb_stats

({(11,
   'ступеням',
   'степеницама'): {'morph_comparison': {'ROOT': {'scores': [1.0],
     'pairs': [('ступен', 'степен')]},
    'SUFF': {'scores': [2.0], 'pairs': [('', 'иц')]},
    'END': {'scores': [2.0],
     'pairs': [('ям', 'ама')]}}, 'overall_stats': {'corr': 0.525,
    'pcorr': 0.35,
    'incorr': 0.125,
    'explained_mistakes': -0.37499999999999994}, 'word_stats': {('лестнице',
     'corr'): ({'ROOT': {'scores': [5.0], 'pairs': [('лест', 'степен')]},
      'SUFF': {'scores': [1.0], 'pairs': [('ниц', 'иц')]},
      'END': {'scores': [3.0], 'pairs': [('е', 'ама')]}},
     0.325),
    ('ступеням',
     'corr'): ({'ROOT': {'scores': [1.0], 'pairs': [('ступен', 'степен')]},
      'SUFF': {'scores': [2.0], 'pairs': [('', 'иц')]},
      'END': {'scores': [2.0], 'pairs': [('ям', 'ама')]}}, 0.15),
    ('платформе',
     'pcorr'): ({'ROOT': {'scores': [7.0], 'pairs': [('платформ', 'степен')]},
      'SUFF': {'scores': [2.0], 'pairs': [('', 'иц')]},
      'END': {'scores': [3.0], 'pa

In [28]:
sk_stats = cognate_stats(slovak_ans, russian_true, slovak_true, sk_cognates, 'latin', num_type)
sk_stats

({(10,
   'повернулся',
   'sa obrátil'): {'morph_comparison': {'PREF': {'scores': [2.0],
     'pairs': [('по', 'об')]},
    'ROOT': {'scores': [3.0], 'pairs': [('вер', 'рат')]},
    'SUFF': {'scores': ['HANDCHECK'], 'pairs': [['н', 'у', 'л'], ['и', 'л']]},
    'POSTFIX': {'scores': [1.0],
     'pairs': [('ся', 'са')]}}, 'overall_stats': {'corr': 0.5217391304347826,
    'pcorr': 0.30434782608695654,
    'incorr': 0.17391304347826086,
    'explained_mistakes': -0.09316770186335402}, 'word_stats': {('повернулся',
     'corr'): ({'PREF': {'scores': [2.0], 'pairs': [('по', 'об')]},
      'ROOT': {'scores': [3.0], 'pairs': [('вер', 'рат')]},
      'SUFF': {'scores': [2.0, 0.0], 'pairs': [('ну', 'и'), ('л', 'л')]},
      'POSTFIX': {'scores': [1.0], 'pairs': [('ся', 'са')]}},
     0.2898550724637681),
    ('обернулся',
     'pcorr'): ({'PREF': {'scores': [1.0], 'pairs': [('о', 'об')]},
      'ROOT': {'scores': [3.0], 'pairs': [('бер', 'рат')]},
      'SUFF': {'scores': ['HANDCHECK'],
       

In [29]:
sn_stats = cognate_stats(slovene_ans, russian_true, slovene_true, sn_cognates, 'latin', num_type)
sn_stats

({(11,
   'ступеням',
   'stopnicam'): {'morph_comparison': {'ROOT': {'scores': [1.6],
     'pairs': [('ступен', 'стопн')]},
    'SUFF': {'scores': [2.0], 'pairs': [('', 'иц')]},
    'END': {'scores': [1.0],
     'pairs': [('ям', 'ам')]}}, 'overall_stats': {'corr': 0.6216216216216216,
    'pcorr': 0.1891891891891892,
    'incorr': 0.1891891891891892,
    'explained_mistakes': -1.0810810810810811}, 'word_stats': {('лестнице',
     'corr'): ({'ROOT': {'scores': [5.0], 'pairs': [('лест', 'стопн')]},
      'SUFF': {'scores': [1.0], 'pairs': [('ниц', 'иц')]},
      'END': {'scores': [2.0], 'pairs': [('е', 'ам')]}},
     0.32432432432432434),
    ('ступеням',
     'corr'): ({'ROOT': {'scores': [1.6], 'pairs': [('ступен', 'стопн')]},
      'SUFF': {'scores': [2.0], 'pairs': [('', 'иц')]},
      'END': {'scores': [1.0], 'pairs': [('ям', 'ам')]}}, 0.21621621621621623),
    ('nan',
     'incorr'): ({'ROOT': {'scores': [5.0], 'pairs': [('', 'стопн')]},
      'SUFF': {'scores': [2.0], 'pairs': [('

In [30]:
import pandas as pd
import numpy as np

def dict_to_dataframe(data):
    rows = []
    
    for key, value in data.items():
        row = {
            'pair': f"{key[0]}, '{key[1]}', '{key[2]}'"
        }
        
        overall = value.get('overall_stats', {})
        row['explained_mistakes'] = overall.get('explained_mistakes', 0.0)
        
        morph = value.get('morph_comparison', {})
        morph_categories = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
        
        for category in morph_categories:
            if category in morph:
                row[category] = morph[category].get('scores', [0.0])
            else:
                row[category] = [0.0]
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    columns_order = ['pair'] + morph_categories + ['explained_mistakes']
    df = df[columns_order]
    
    return df

ukr_true_df = dict_to_dataframe(ukr_stats[0])
ukr_true_df

,pair,PREF,ROOT,LINK,SUFF,END,POSTFIX,explained_mistakes
0,"10, 'повернулся', 'повернувся'",[0.0],[0.0],[0.0],"[0.0, 0.0, 1.0]",[0.0],[0.0],0.000000
1,"12, 'шашек', 'шашок'",[0.0],[0.0],[0.0],[1.0],[0.0],[0.0],-2.087719
2,"13, 'спиною', 'спиною'",[0.0],[0.0],[0.0],[0.0],[0.0],[0.0],0.000000
3,"14, 'финики', 'фініки'",[0.0],[0.4],[0.0],[0.0],[0.0],[0.0],0.748284
4,"16, 'вырвался', 'вирвався'",[1.0],[0.0],[0.0],"[0.0, 1.0]",[0.0],[0.0],0.768421
...,...,...,...,...,...,...,...,...
121,"630, 'заговорил', 'заговорив'",[0.0],[0.0],[0.0],"[0.0, 1.0]",[0.0],[0.0],0.000000
122,"631, 'присутствовал', 'присутній'",[0.0],[3.0],[0.0],[HANDCHECK],[2.0],[0.0],0.000000
123,"632, 'балконе', 'балконі'",[0.0],[0.0],[0.0],[0.0],[0.6],[0.0],0.867868
124,"633, 'тайно', 'таємно'",[0.0],[2.0],[0.0],[0.0],[0.0],[0.0],0.817048


In [31]:
ukr_true_df.to_csv(
    'ukr_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [32]:
bel_true_df = dict_to_dataframe(bel_stats[0])
bel_true_df.to_csv(
    'bel_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

cz_true_df = dict_to_dataframe(cz_stats[0])
cz_true_df.to_csv(
    'cz_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
pol_true_df = dict_to_dataframe(pol_stats[0])
pol_true_df.to_csv(
    'pol_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
bg_true_df = dict_to_dataframe(bg_stats[0])
bg_true_df.to_csv(
    'bg_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sn_true_df = dict_to_dataframe(sn_stats[0])
sn_true_df.to_csv(
    'sn_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sk_true_df = dict_to_dataframe(sk_stats[0])
sk_true_df.to_csv(
    'sk_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sb_true_df = dict_to_dataframe(sb_stats[0])
sb_true_df.to_csv(
    'sb_true.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [33]:
import pandas as pd
import numpy as np

def alternate_to_dataframe(data):
    rows = []
    
    for key, value in data.items():
        code = key[0]
        original = key[2]
        overall = value.get('overall_stats', {})
        explained_mistakes = overall.get('explained_mistakes', 0.0)
        for key1, value1 in value['word_stats'].items():
            row = {
                'pair': f"{code}, '{key1[0]}', '{original}'"
            }
            row['explained_mistakes'] = explained_mistakes
            morph = value1[0]
            morph_categories = ['PREF', 'ROOT', 'LINK', 'SUFF', 'END', 'POSTFIX']
            
            for category in morph_categories:
                if category in morph:
                    row[category] = morph[category].get('scores', [0.0])
                else:
                    row[category] = [0.0]
            
            rows.append(row)
    
    df = pd.DataFrame(rows)
    
    columns_order = ['pair'] + morph_categories + ['explained_mistakes']
    df = df[columns_order]
    
    return df

ukr_part1_df = alternate_to_dataframe(ukr_stats[1])
ukr_part1_df

,pair,PREF,ROOT,LINK,SUFF,END,POSTFIX,explained_mistakes
0,"11, 'ступенькам', 'сходинок'",[1.0],[3.6],[0.0],[HANDCHECK],[2.0],[0.0],-1.605263
1,"11, 'nan', 'сходинок'",[1.0],[3.0],[0.0],[4.0],[0.0],[0.0],-1.605263
2,"11, 'ступеням', 'сходинок'",[1.0],[5.6],[0.0],[4.0],[2.0],[0.0],-1.605263
3,"11, 'сходням', 'сходинок'",[0.0],[0.0],[0.0],[HANDCHECK],[2.0],[0.0],-1.605263
4,"11, 'лестнице', 'сходинок'",[1.0],[3.6],[0.0],[HANDCHECK],[1.0],[0.0],-1.605263
...,...,...,...,...,...,...,...,...
788,"628, 'прилипли', 'понахилялися'",[HANDCHECK],[2.0],[0.0],[HANDCHECK],[0.0],[2.0],-1.972973
789,"628, 'граждане', 'понахилялися'",[4.0],[4.6],[0.0],[HANDCHECK],[0.6],[2.0],-1.972973
790,"628, 'nan', 'понахилялися'",[4.0],[3.0],[0.0],[2.0],[1.0],[2.0],-1.972973
791,"628, 'что понаехали', 'понахилялися'",[0.0],[0.0],[0.0],[0.0],[0.0],[0.0],-1.972973


In [34]:
ukr_part1_df.to_csv(
    'ukr_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

bel_part1_df = alternate_to_dataframe(bel_stats[1])
bel_part1_df.to_csv(
    'bel_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [35]:
bg_part1_df = alternate_to_dataframe(bg_stats[1])
bg_part1_df.to_csv(
    'bg_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
pol_part1_df = alternate_to_dataframe(pol_stats[1])
pol_part1_df.to_csv(
    'pol_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
cz_part1_df = alternate_to_dataframe(cz_stats[1])
cz_part1_df.to_csv(
    'cz_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sb_part1_df = alternate_to_dataframe(sb_stats[1])
sb_part1_df.to_csv(
    'sb_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sn_part1_df = alternate_to_dataframe(sn_stats[1])
sn_part1_df.to_csv(
    'sn_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)
sk_part1_df = alternate_to_dataframe(sk_stats[1])
sk_part1_df.to_csv(
    'sk_part11.csv',
    index=False,
    encoding='utf-8',
    sep=',',
    header=True
)

In [36]:
def transform_morpheme_dataframe(filepath):
    
    df = pd.read_csv(filepath)

    def safe_eval_list(val, default_len=3):
        if pd.isna(val) or val == '' or val == '[]':
            return [0] * default_len
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return parsed + [0] * (default_len - len(parsed))
                else:
                    return [parsed] + [0] * (default_len - 1)
            elif isinstance(val, list):
                return val + [0] * (default_len - len(val))
            else:
                return [val] + [0] * (default_len - 1)
        except:
            return [0] * default_len
    
    def safe_eval_single(val):
        if pd.isna(val) or val == '' or val == '[]':
            return 0
        try:
            if isinstance(val, str):
                val = val.replace("'", '"')
                parsed = ast.literal_eval(val)
                if isinstance(parsed, list):
                    return parsed[0] if parsed else 0
                else:
                    return parsed
            else:
                return val
        except:
            return 0
    
    new_data = []
    
    for idx, row in df.iterrows():
        new_row = {
            'pair': row['pair'],
            'explained_mistakes': row['explained_mistakes']
        }
        
        pref_list = safe_eval_list(row['PREF'], 2)
        new_row['PREF1'] = pref_list[0]
        new_row['PREF2'] = pref_list[1]
        
        root_list = safe_eval_list(row['ROOT'], 2)
        new_row['ROOT1'] = root_list[0]
        new_row['ROOT2'] = root_list[1]

        new_row['LINK'] = safe_eval_single(row['LINK'])
        
        suff_list = safe_eval_list(row['SUFF'], 3)
        new_row['SUFF1'] = suff_list[0]
        new_row['SUFF2'] = suff_list[1]
        new_row['SUFF3'] = suff_list[2]
        
        new_row['END'] = safe_eval_single(row['END'])
        
        new_row['POSTFIX'] = safe_eval_single(row['POSTFIX'])
        
        new_data.append(new_row)
    
    new_df = pd.DataFrame(new_data)
    
    columns_order = ['pair', 'PREF1', 'PREF2', 'ROOT1', 'ROOT2', 'LINK', 
                     'SUFF1', 'SUFF2', 'SUFF3', 'END', 'POSTFIX', 
                     'explained_mistakes']
    new_df = new_df[columns_order]
    
    return new_df